# Model Training & Batch Transform DAG
This is the core ML pipeline. We pull data from our **Feature Store**, train a Random Forest on SageMaker infrastructure, and execute a **Batch Transform** job to generate predictions.

In [ ]:
!pip install "sagemaker<3.0.0" awswrangler

import os
import boto3
import sagemaker
import pandas as pd
import numpy as np
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup
import awswrangler as wr
import time

# Setup
role = sagemaker.get_execution_role()
sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "aai-540-group6/nyc-collisions-ml"

print(f"Executing as Role: {role}")

### 1. Retrieve Data from Feature Store
Run an Athena query to grab the engineered features from our Offline Store.

In [ ]:
# UPDATE THIS: Name of the feature group created in Notebook 03
fg_name = "REPLACE_WITH_YOUR_FEATURE_GROUP_NAME"

fg = FeatureGroup(name=fg_name, sagemaker_session=sess)
query = fg.athena_query()
db = query.database
tbl = query.table_name

sql = f'SELECT * FROM "{db}"."{tbl}"'
print(f"Pulling data via: {sql}")

df = wr.athena.read_sql_query(sql, database=db)
print(f"Retrieved {df.shape[0]} rows.")

### 2. Prepare the Splits
We need a 3-way split: Training, Validation, and a Batch inference set (without the target label).

In [ ]:
features = ['borough', 'month', 'hour', 'is_rush_hour', 'is_weekend', 'cause_category', 'vehicle_type']
target = 'target'
id_col = 'collision_id'

# Random split
rand = np.random.rand(len(df))
train_idx = rand < 0.8
val_idx = (rand >= 0.8) & (rand < 0.9)
batch_idx = rand >= 0.9

df_train = df[train_idx][features + [target]]
df_val = df[val_idx][features + [target]]
df_batch = df[batch_idx][features + [id_col]] # IDs kept for joining

print(f"Splits -> Train: {len(df_train)}, Val: {len(df_val)}, Batch: {len(df_batch)}")

### 3. Upload to S3

In [ ]:
os.makedirs('data_splits', exist_ok=True)
df_train.to_csv('data_splits/train.csv', index=False, header=False)
df_val.to_csv('data_splits/val.csv', index=False, header=False)
df_batch.to_csv('data_splits/batch.csv', index=False, header=False)

s3_train = sess.upload_data('data_splits/train.csv', bucket, f"{prefix}/train")
s3_val = sess.upload_data('data_splits/val.csv', bucket, f"{prefix}/validation")
s3_batch = sess.upload_data('data_splits/batch.csv', bucket, f"{prefix}/batch")

### 4. Training (SKLearn Estimator)
Kick off a training job on a dedicated ml.m5.xlarge instance.

In [ ]:
from sagemaker.sklearn.estimator import SKLearn

est = SKLearn(
    entry_point='sagemaker_train.py',
    source_dir='../src',
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    framework_version='1.2-1',
    py_version='py3',
    hyperparameters={'n-estimators': 100, 'max-depth': 12}
)

est.fit({'train': s3_train})
print("Model fit complete.")

### 5. Batch Inferences (I/O Joining)
Execute the transform job and join predictions with their IDs.

In [ ]:
trans = est.transformer(
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f"s3://{bucket}/{prefix}/predictions",
    assemble_with='Line'
)

trans.transform(
    data=s3_batch,
    content_type='text/csv', 
    split_type='Line',
    input_filter='$[1:]', # Send only features to model
    join_source='Input',
    output_filter='$[0,-1]' # Output ID and result
)

trans.wait()
print("Predictions ready in S3.")

### 6. Final Audit
Download and check the results.

In [ ]:
res = wr.s3.read_csv(f"s3://{bucket}/{prefix}/predictions/batch.csv.out", header=None)
res.columns = ['id', 'pred']
display(res.head())